# Multimodal Price Predictor — end-to-end run

Dual-encoder (EmbeddingGemma + SigLIP2) fusion model for the Amazon ML Challenge
2025 style price prediction task. This notebook runs the **entire pipeline**
(data → preprocessing → embeddings → training → evaluation → predictions)
with **no local GPU required**.

**Environment assumptions**
- Designed for **Kaggle Notebooks** (free T4/P100, ~30 hrs/week) or Google Colab.
  Also works unchanged as the entrypoint for a Hugging Face Job.
- This notebook expects the project repo (the `src/`, `scripts/`, `configs/`
  folders) to sit in the **same working directory** as this notebook — e.g.
  upload the whole `price-predictor/` folder as a Kaggle Dataset/Notebook
  input, or unzip it into the Colab/Job working directory before running.
- Data is pulled directly from a Kaggle-hosted mirror of the competition
  dataset inside this notebook — it never needs to touch your laptop.
- GPU is required for stage 02 (embedding extraction). Stages 01, 03, 04, 05
  are fast enough to run on CPU if a GPU session isn't available, though
  stage 03 will also benefit from GPU if present.

In [ ]:
import subprocess, sys

def sh(cmd):
    """Run a shell command, streaming output, and raise on failure so a
    failed stage stops the notebook instead of silently continuing to the
    next cell with missing data."""
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected — stage 02 (embedding extraction) will be slow. "
          "On Kaggle: Settings > Accelerator > GPU T4 x2.")


## 1. Install dependencies

In [ ]:
sh("pip install -q -r requirements.txt")


## 2. Verify the repo structure is present

This notebook does not contain the project code itself — it orchestrates the
`scripts/` you already have. If this cell fails, the repo folder was not
uploaded/mounted correctly.

In [ ]:
from pathlib import Path

REQUIRED_PATHS = [
    "src/encoders/registry.py",
    "src/fusion/registry.py",
    "src/models/price_model.py",
    "scripts/01_preprocess.py",
    "scripts/02_extract_embeddings.py",
    "scripts/03_train.py",
    "scripts/04_evaluate.py",
    "scripts/05_predict.py",
    "configs/base.yaml",
]

missing = [p for p in REQUIRED_PATHS if not Path(p).exists()]
if missing:
    raise FileNotFoundError(
        "Missing expected project files: " + ", ".join(missing) +
        "\nMake sure the price-predictor/ repo contents are in the current working directory."
    )
print("Repo structure OK.")


## 3. Get the dataset

Pulls the Kaggle-hosted mirror of the Amazon ML Challenge 2025 (Smart Product
Pricing) dataset directly into this environment. Requires a Kaggle API token
(`kaggle.json`) — on Kaggle Notebooks this is automatic; on Colab, upload your
token first.

**Cross-check before running:** confirm the dataset slug below still resolves
— dataset mirrors occasionally move. Search "amazon ml challenge 2025 price
prediction" on kaggle.com/datasets if this specific slug 404s.

In [ ]:
import os

DATASET_SLUG = "suvroo/amazon-ml"  # verify this still resolves before running
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

try:
    import kagglehub
    dataset_path = kagglehub.dataset_download(DATASET_SLUG)
    print("Downloaded to:", dataset_path)

    for csv_name in ["train.csv", "test.csv"]:
        src = Path(dataset_path) / csv_name
        if src.exists():
            sh(f"cp '{src}' '{RAW_DIR / csv_name}'")
        else:
            print(f"WARNING: {csv_name} not found at {src} — check the dataset's actual file names "
                  f"and copy manually into {RAW_DIR}/")
except ImportError:
    print("kagglehub not installed. Install with: pip install kagglehub, "
          f"or manually place train.csv/test.csv into {RAW_DIR}/")
except Exception as e:
    print(f"Automatic download failed ({e}). "
          f"Manually place train.csv and test.csv into {RAW_DIR}/ and re-run this cell to verify.")

for f in ["train.csv", "test.csv"]:
    p = RAW_DIR / f
    print(f, "present" if p.exists() else "MISSING")


## 4. Stage 01 — Preprocess

Cleans text, downloads + resizes all product images. This is the network-bound
stage — expect this to be the slowest single step if bandwidth is limited.

In [ ]:
sh("python scripts/01_preprocess.py --config configs/base.yaml")


## 5. Stage 02 — Extract embeddings (GPU stage)

Runs the frozen EmbeddingGemma + SigLIP2 encoders once and caches the results.
This is the only stage that meaningfully needs a GPU.

In [ ]:
sh("python scripts/02_extract_embeddings.py --config configs/base.yaml")


## 6. Stage 03 — Train fusion + regression head

Trains only the small trainable component (~50M params) on cached embeddings.
Fast regardless of GPU/CPU since the heavy backbones are not involved here.

In [ ]:
sh("python scripts/03_train.py --config configs/base.yaml")


## 7. Stage 04 — Evaluate on held-out split

In [ ]:
sh("python scripts/04_evaluate.py --config configs/base.yaml")

import json
report_paths = list(Path("reports").glob("*/eval_report.json"))
if report_paths:
    with open(report_paths[0]) as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("No eval report found — check stage 04 output above for errors.")


## 8. Stage 05 — Predict on the test set / generate submission

In [ ]:
sh("python scripts/05_predict.py --config configs/base.yaml --output data/raw/test_out.csv")

import pandas as pd
preds = pd.read_csv("data/raw/test_out.csv")
print(preds.head())


## 9. Quick sanity check — single live prediction

Runs the full end-to-end chain (encoders + fusion + head together, not cached
embeddings) on one example, the same code path the API/demo uses.

In [ ]:
import sys
sys.path.insert(0, ".")

from src.inference.predictor import Predictor
from src.utils.config import load_config
from src.utils.exceptions import PricePredictorError

try:
    config = load_config("configs/base.yaml")
    checkpoint_path = f"{config['checkpoint_dir']}/best.pt"
    predictor = Predictor(config, checkpoint_path)

    sample_text = "Wireless noise-canceling headphones, 30hr battery, over-ear, black"
    sample_image_url = "https://m.media-amazon.com/images/I/example.jpg"  # replace with a real URL

    price = predictor.predict_one(sample_text, sample_image_url)
    print(f"Predicted price: ${price:.2f}")
except PricePredictorError as e:
    print(f"Prediction failed: {e}")
